In [3]:
!pip install rotary-embedding-torch
!wget -O sonnets_oupoco_tei.xml "https://zenodo.org/record/5646940/files/sonnets_oupoco_tei.xml?download=1"

--2026-04-13 18:20:54--  https://zenodo.org/record/5646940/files/sonnets_oupoco_tei.xml?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.52.235, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/5646940/files/sonnets_oupoco_tei.xml [following]
--2026-04-13 18:20:54--  https://zenodo.org/records/5646940/files/sonnets_oupoco_tei.xml
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 10323394 (9.8M) [application/octet-stream]
Saving to: ‘sonnets_oupoco_tei.xml’

sonnets_oupoco_tei. 100%[===================>]   9.84M  7.19MB/s    in 1.4s    

2026-04-13 18:20:56 (7.19 MB/s) - ‘sonnets_oupoco_tei.xml’ saved [10323394/10323394]



In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from rotary_embedding_torch import RotaryEmbedding
from transformers import get_cosine_schedule_with_warmup
import math
import itertools
import matplotlib.pyplot as plt
import math
from collections import OrderedDict
import pandas as pd
import json
import re

In [5]:
class TransformerBlock(nn.Module):
  def __init__(self, d_model, n_heads, d_ff, dropout=0.1, theta=1e4):
      super().__init__()

      self.d_model = d_model
      self.n_heads = n_heads
      self.head_dim = d_model // self.n_heads
      self.dropout = dropout

      self.projection = nn.Linear(d_model, d_model * 3)
      self.out = nn.Linear(d_model, d_model)

      self.rope = RotaryEmbedding(dim=self.head_dim, theta=theta)

      self.ff = nn.Sequential(
          nn.Linear(d_model, d_ff),
          nn.GELU(),
          nn.Linear(d_ff, d_model)
      )

      self.ln1 = nn.LayerNorm(d_model)
      self.ln2 = nn.LayerNorm(d_model)
      self.drop = nn.Dropout(dropout)


  def forward(self, x):
      attention_drop = self.dropout if self.training else 0.0
      batch_size = x.size(0)
      context_len = x.size(1)

      h = self.ln1(x)  # (batch_size, context_len, d_model)
      QKV = self.projection(h)  # (batch_size, context_len, d_model*3)
      Q, K, V = QKV.chunk(3, dim=-1)  # (batch_size, content_len, d_model)

      Q = Q.view(batch_size, context_len, self.n_heads,
                 self.head_dim).transpose(1,2)  # (batch_size, n_heads, context_len, head_dim)

      K = K.view(batch_size, context_len, self.n_heads,
                 self.head_dim).transpose(1,2)

      V = V.view(batch_size, context_len, self.n_heads,
                 self.head_dim).transpose(1,2)

      Q = self.rope.rotate_queries_or_keys(Q)
      K = self.rope.rotate_queries_or_keys(K)

      context = F.scaled_dot_product_attention(
                Q, K, V,
                attn_mask=None,
                dropout_p=attention_drop,
                is_causal=True)

      context = context\
                .transpose(1, 2)\
                .contiguous()\
                .view(batch_size, context_len, self.d_model)

      x = x + self.drop(self.out(context))
      x = x + self.drop(self.ff(self.ln2(x)))

      return x


In [6]:
class AIexandrin(nn.Module):
    def __init__(self,
                 vocab_size,
                 d_model=256,
                 n_heads=8,
                 d_ff=1024,
                 n_layers=6,
                 dropout=0.1):

        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.n_layers = n_layers

        self.token_emb = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)


    def forward(self, x):
        batch_size, context_len = x.size()

        h = self.token_emb(x)  # (batch_size, context_len, d_model)

        for layer in self.layers:
            h = layer(h)

        h = self.ln_f(h)
        logits = self.head(h)  # (batch_size, context_len, vocab_size)

        return logits

In [7]:
import xml.etree.ElementTree as ET
import os


tei_file = "/kaggle/working/sonnets_oupoco_tei.xml"
output_file = "dataset_poesie.txt"
ns = {"tei": "http://www.tei-c.org/ns/1.0"}


tree = ET.parse(tei_file)
root = tree.getroot()

dataset_content = []
poem_count = 0

for text_el in root.findall(".//tei:text", ns):
    for div in text_el.findall(".//tei:div", ns):
        div_type = div.get("type", "").lower()
        strophes = []
        for lg in div.findall("tei:lg", ns):
            lines = []
            for l in lg.findall("tei:l", ns):
                line_text = "".join(l.itertext()).strip()
                if line_text:
                    lines.append(line_text)

            if lines:
                strophes.append("\n".join(lines))

        if strophes:
            poem_body = "\n\n".join(strophes)
            full_entry = f"§\n{poem_body}\n¤"
            dataset_content.append(full_entry)
            poem_count += 1

final_string = "\n\n".join(dataset_content)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_string)


file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
char_count = len(final_string)
vocab = sorted(list(set(final_string)))

print(f"File : {output_file}")
print(f"Poem count : {poem_count}")
print(f"Size : {file_size_mb:.2f} Mo")


File : dataset_poesie.txt
Poem count : 4870
Size : 3.01 Mo


In [8]:
def clean_data(filename):
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    replacements = {
        "–": "-",
        "—": "-",
        "«": '"',
        "»": '"',
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    valid = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
                "àâæçèéêëîïôöùûüÿœÀÂÇÈÉÊÔÙŒ"
                " \n\"',-.:;!?()§¤")

    text = "".join(c for c in text if c in valid)
    return text


text = clean_data("/kaggle/working/dataset_poesie.txt")
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

In [9]:
def encode(s):
    return [char_to_idx[c] for c in s]


def decode(l):
    return "".join([idx_to_char[i] for i in l])


import re
poems = re.findall(r"§.*?¤", text, re.DOTALL)
n = int(0.9 * len(poems))
train = torch.tensor(encode("\n\n".join(poems[:n])), dtype=torch.long)
dev = torch.tensor(encode("\n\n".join(poems[n:])), dtype=torch.long)

print(f"train : {n}\ndev : {len(poems) - n}")

train : 4383
dev : 487


In [10]:
def generate(model, prompt, max_new_tokens=700, context_len=512, temperature=0.8):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to("cuda")

        eos_sequence = encode("¤")

        for _ in range(max_new_tokens):
            x_crop = x[:, -context_len:]
            logits = model(x_crop)
            logits = logits[:, -1, :] / temperature

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)

            if x[0].tolist()[-len(eos_sequence):] == eos_sequence:
                break
        
        result = decode(x[0].tolist())
        model.train()
        return result

In [11]:
def get_batch(split, context_len=512, batch_size=64):
    data = train if split == "train" else dev
    ix = torch.randint(len(data) - context_len, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])
    return x.to("cuda"), y.to("cuda")


def evaluate(model, context_len=512, batch_size=64):
    model.eval()
    running_loss = 0.0
    total = 0
    n_blocks = len(dev) // context_len
    x = dev[:n_blocks * context_len].view(n_blocks, context_len)
    y = dev[1:n_blocks * context_len + 1].view(n_blocks, context_len)

    with torch.no_grad():
        for i in range(0, n_blocks, batch_size):
            X_dev = x[i:i+batch_size].to("cuda")
            Y_dev = y[i:i+batch_size].to("cuda")
            logits = model(X_dev)
            loss = F.cross_entropy(logits.view(-1, vocab_size), Y_dev.view(-1), reduction="sum")
            running_loss += loss.item()
            total += X_dev.size(0) * context_len

    model.train()
    return running_loss / total

In [13]:
contexts = [256, 512]
LR = [1e-3, 3e-4]
results = []
steps = 20_000
patience = 5

for context_len, lr in itertools.product(contexts, LR):
    print(f"\n ###### LR {lr}, context size {context_len} #####")
    torch.cuda.empty_cache()

    model = nn.DataParallel(AIexandrin(vocab_size=vocab_size, dropout=0.2)).to("cuda")
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=200, num_training_steps=steps)
    loss_fn = nn.CrossEntropyLoss()

    metrics = {"steps": [], "train_loss": [], "dev_loss": [], "bpc": []}
    best_dev_loss = float("inf")
    best_train_loss = float("inf")
    best_step = 0
    patience_count = 0
    running_loss = 0.0
    total = 0

    for step in range(steps):
        model.train()
        x, y = get_batch("train", context_len=context_len)
        logits = model(x)
        loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * x.size(0) * context_len
        total += x.size(0) * context_len

        if step % 200 == 0:
            train_loss = running_loss / total
            dev_loss = evaluate(model, context_len=context_len)
            bpc = dev_loss / math.log(2)

            running_loss = 0.0
            total = 0

            metrics["steps"].append(step)
            metrics["train_loss"].append(train_loss)
            metrics["dev_loss"].append(dev_loss)
            metrics["bpc"].append(bpc)

            print(f"step {step} | train {train_loss:.4f} | dev {dev_loss:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | bpc {bpc:.4f}")

            if dev_loss < best_dev_loss:
                best_dev_loss = dev_loss
                best_train_loss = train_loss
                best_step = step
                patience_count = 0
                torch.save(model.module.state_dict(), f"model_lr{lr}_context_len{context_len}.pt")
            else:
                patience_count += 1
                if patience_count >= patience:
                    print(f"Early stopping at step {step}")
                    break

        if step % 500 == 0:
            print(generate(model, prompt="§\n", max_new_tokens=700, context_len=context_len, temperature=0.8))

    else:
        train_loss = running_loss / total if total > 0 else float("nan")
        dev_loss = evaluate(model, context_len=context_len)
        print(f"train {train_loss:.4f} | dev {dev_loss:.4f}")
        print(generate(model, prompt="§\n", max_new_tokens=700, context_len=context_len, temperature=0.8))

    results.append({
        "initial_lr": lr,
        "context_len": context_len,
        "metrics": metrics,
        "best_dev_loss": best_dev_loss,
        "best_train_loss": best_train_loss,
        "best_step": best_step
    })

print("\nResults:")
for r in sorted(results, key=lambda x: x["best_dev_loss"]):
    print(f"lr={r['initial_lr']} ctx={r['context_len']} | step {r['best_step']} | train {r['best_train_loss']:.4f} | dev {r['best_dev_loss']:.4f}")


 ###### LR 0.001, context size 256 #####
step 0 | train 4.7049 | dev 4.7043 | lr 5.00e-06 | bpc 6.7868
§
ÇZuXWQ§uA¤
step 200 | train 2.4132 | dev 1.7965 | lr 1.00e-03 | bpc 2.5918
step 400 | train 1.6981 | dev 1.5362 | lr 1.00e-03 | bpc 2.2162
§
Dans ton prompagne au sein de soudre se mien infumie
Ce que le lèvre le thimps de son esprit je suis,
Dans la boise de tout ingéner le tant qui fuit les routes.

La fleur au vasque insoir, et tout était mes plaines
Et devant se répant, le souffre du sombre ;
D'un sombre du sauveur de ma par de votre nom pas,

Et que la scerche dans le champ de l'apporte,
Ce que l'arrêt de trouvers l'esprit dont sa mort,
En pris de cet aimé ! ce qui sent plus qui marche
Sans le marc de pied qui se ploin de son cœur,

A ton d'homme ainsent d'un pas ses mots ;
C'est au fuit, le pour mon rose en ses chants,
De la mer eÀ de l'on jusqu'à l'insuit mon rêve ;
Nul en fait le seine chaque sans cesse en attrique,
J'ai 
step 600 | train 1.5331 | dev 1.4428 | lr 9.99e-04 |

KeyboardInterrupt: 

In [ ]:
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters : {params}")

In [ ]:
fig1, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, r in enumerate(results):
    ax = axes[i]
    ax.plot(r["metrics"]["steps"], r["metrics"]["train_loss"], label="train loss")
    ax.plot(r["metrics"]["steps"], r["metrics"]["dev_loss"], label="dev loss")
    ax.set_title(f"lr={r['initial_lr']}, context={r['context_len']}")
    ax.set_xlabel("steps")
    ax.set_ylabel("loss")
    ax.legend()

fig1.tight_layout()
plt.show()


fig2, ax = plt.subplots(figsize=(10, 5))

for r in results:
    label = f"lr={r['initial_lr']}, ctx={r['context_len']}"
    ax.plot(r["metrics"]["steps"], r["metrics"]["bpc"], label=label)

ax.set_title("BPC (dev)")
ax.set_xlabel("steps")
ax.set_ylabel("BPC")
ax.legend()
fig2.tight_layout()
plt.show()

In [12]:
def load_model(model_path):
    model = AIexandrin(vocab_size=vocab_size, dropout=0.2).to("cuda")
    path_to_model = model_path
    state_dict = torch.load(path_to_model, map_location="cuda")
    
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k.startswith("module."):
            name = k[7:]  # to remove module. because of nn.DataParallel
        else:
            name = k
        new_state_dict[name] = v

    model.load_state_dict(new_state_dict)
    model.eval()
    print("Loaded model")
    return model

In [13]:
model_paths = [
    "/kaggle/input/models/axelmeunier/aiexandrin-ctxt512/pytorch/default/1/model_lr0.001_context_len512.pt",
    "/kaggle/input/models/axelmeunier/aiexandrin-ctxt256/pytorch/default/1/model_lr0.001_context_len256.pt"
]

results = []

for path in model_paths:
    model = load_model(path)
    
    model_name = path.split("/")[-1]
    
    for i in range(20):
        text = generate(model, prompt="§\n", max_new_tokens=700, temperature=0.8)
        
        results.append({
            "model": model_name,
            "id": i,
            "poem": text
        })
        if i % 5 == 0:
            print(f"step {i}")
    print(f"{model_name} done")
    
df = pd.DataFrame(results)
print(df.sample())


with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f)

Loaded model
step 0
step 5
step 10
step 15
model_lr0.001_context_len512.pt done
Loaded model
step 0
step 5
step 10
step 15
model_lr0.001_context_len256.pt done
                              model  id  \
28  model_lr0.001_context_len256.pt   8   

                                                 poem  
28  §\nPuissement son amour souci de la cité sur l...  


In [19]:
def analyse_structure(poem):
    pattern = r"^§\n(?:[^\n]+\n){4}\n(?:[^\n]+\n){4}\n(?:[^\n]+\n){3}\n(?:[^\n]+\n){3}¤$"
    return bool(re.match(pattern, poem))

In [21]:
df = pd.read_json("/kaggle/working/results.json")
df.query("model=='model_lr0.001_context_len256.pt'")["poem"].str.len().agg(["mean", "min", "max", "std"])

mean    571.950000
min      94.000000
max     702.000000
std     203.593888
Name: poem, dtype: float64

In [25]:
df = pd.read_json("/kaggle/working/results.json")

df["structure_ok"] = df["poem"].apply(analyse_structure)

res = df.groupby("model")["structure_ok"].agg(["sum", "mean"]).reset_index()
res.columns = ["model", "count", "%"]
res["%"] = res["%"] * 100

print(res)

                             model  count     %
0  model_lr0.001_context_len256.pt      0   0.0
1  model_lr0.001_context_len512.pt     14  70.0


In [17]:
with open("/kaggle/working/dataset_poesie.txt", "r", encoding="utf-8") as f:
    raw = f.read()

text = clean_data("/kaggle/working/dataset_poesie.txt")
poems_corpus = re.findall(r"§.*?¤", text, re.DOTALL)
corpus_lengths = pd.Series([len(p) for p in poems_corpus])
print(corpus_lengths.agg(["mean", "min", "max", "std"]))

mean    628.940041
min      81.000000
max     759.000000
std      82.344687
dtype: float64


In [30]:
model = load_model("/kaggle/input/models/axelmeunier/aiexandrin-ctxt512/pytorch/default/1/model_lr0.001_context_len512.pt")
print(generate(model, prompt="§\n", max_new_tokens=700, temperature=0.8))

Loaded model
§
Contrainte aux forêts sur les flots bleus,
Et sur les chemins d'or les limpides roseaux,
Où l'argile mouvant se monte aux abeilles ;
D'une rose croît tout embaumée en deuil brunis.

Dans les hauts bras venus, les clairs conseilleux
(Pour calmer les plis du sein des clartés vermeilles,
Immobile encor des forêts d'argent et de brouillards.
J'aime, se lassa d'aile, et les fleurs à l'aigle.

Ils couriront toujours ; tous, les voici les airs
Pour l'aigle de l'autel qu'ils sont temps de soie;
Mais la tristesse endort dans la lambe aussitôt !

Leur coeur enlace encor perdre ce fils de joie !
Et ce soir ils me sentient, l'artiste et l'on guérit,
Revivraient dans ton sein leur pureté constante.
¤
